# Machine Learning - Ev Tapşırığı
**Mövzu:** Data Preprocessing ve Regression Metrics  


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Hazirdir")

Hazirdir


## 1. One-Hot Encoding vs Factorize

### One-Hot Encoding nedir ve nece işleyir?
Maşın öyrenmesi modelleri reqemlerle işleyir, ona göre "Qırmızı", "Yaşıl", "Mavi" kimi kateqorik (metn) deyerleri reqeme çevirmek lazımdır. One-Hot Encoding her unikal kateqoriya üçün ayrıca bir sütun yaradır. Her setirde yalnız o kateqoriyaya uyğun sütunda 1, qalan sütunlarda 0 olur. Meselen 3 reng varsa, 3 yeni sütun yaranır. Bu yanaşmada kateqoriyalar arasında saxta bir sıralama yaranmır: model "Mavi, Qırmızıdan böyükdür" kimi yanlış netice çıxarmır. Dezavantajı odur ki, kateqoriya sayı çox olanda (meselen 1000 şeher) sütun sayı keskin artır.

### pd.factorize() nedir?
`pd.factorize()` her unikal deyere bir tam eded (0, 1, 2, ...) verir ve bir sütunda saxlayır. Yeni sütun yaranmır, yalnız deyerler ededlere çevrilir. Funksiya iki şey qaytarır: kodlanmış massiv ve unikal deyerlerin siyahısı. Çatışan (NaN) deyerlere default olaraq -1 verilir.

### Aralarındaki esas ferqler
One-Hot çox sütun yaradır, factorize ise tek sütun saxlayır. One-Hot kateqoriyalar arasında sıra iddia etmir, factorize ise 0 < 1 < 2 kimi saxta sıra yaradır. Bu sebebden factorize xetti modellerde (Linear Regression, Logistic Regression, KNN, neyron şebekeler) yanlış netice verebilir, çünki model ededlerin böyüklüyünü mena kimi qebul edir. Factorize yaddaş baxımından qenaetlidir ve sıra anlayışı olmayan agac esaslı modellerde (Decision Tree, Random Forest, XGBoost) adeten problem yaratmır.

### Hansı halda hansını seçmeli?
Kateqoriyalar arasında tebii sıra yoxdursa (reng, şeher, mehsul növü) ve model xetti/mesafe esaslıdırsa, One-Hot seçilmelidir. Kateqoriya sayı çox böyükdürse ve agac esaslı model işledilirse, factorize (ve ya Label/Ordinal Encoding) meqsedeuyğundur. Kateqoriyalar tebii sıraya malikdirse (Aşağı < Orta < Yüksek), sırani özümüz verdiyimiz OrdinalEncoder daha doğrudur. Target (hedef) sütununu kodlamaq üçün factorize rahatdır.

In [2]:
df = pd.DataFrame({
    "Reng": ["Qirmizi", "Yasil", "Mavi", "Yasil", "Qirmizi"],
    "Qiymet": [10, 20, 15, 22, 11]
})
print("Original:")
print(df, "\n")

# --- One-Hot Encoding (pandas) ---
onehot_pd = pd.get_dummies(df, columns=["Reng"], dtype=int)
print("pd.get_dummies:")
print(onehot_pd, "\n")

# --- One-Hot Encoding (sklearn) ---
enc = OneHotEncoder(sparse_output=False)
arr = enc.fit_transform(df[["Reng"]])
print("sklearn OneHotEncoder kateqoriyalar:", enc.categories_[0])
print(pd.DataFrame(arr, columns=enc.get_feature_names_out()), "\n")

# --- factorize ---
codes, uniques = pd.factorize(df["Reng"])
print("factorize kodlari :", codes)
print("factorize unikallar:", list(uniques), "\n")

df["Reng_kod"] = codes
print(df)

# --- NaN ile factorize ---
s = pd.Series(["A", "B", None, "A"])
print("\nNaN ile factorize:", pd.factorize(s)[0], "(NaN = -1)")

Original:
      Reng  Qiymet
0  Qirmizi      10
1    Yasil      20
2     Mavi      15
3    Yasil      22
4  Qirmizi      11 

pd.get_dummies:
   Qiymet  Reng_Mavi  Reng_Qirmizi  Reng_Yasil
0      10          0             1           0
1      20          0             0           1
2      15          1             0           0
3      22          0             0           1
4      11          0             1           0 

sklearn OneHotEncoder kateqoriyalar: ['Mavi' 'Qirmizi' 'Yasil']
   Reng_Mavi  Reng_Qirmizi  Reng_Yasil
0        0.0           1.0         0.0
1        0.0           0.0         1.0
2        1.0           0.0         0.0
3        0.0           0.0         1.0
4        0.0           1.0         0.0 

factorize kodlari : [0 1 2 1 0]
factorize unikallar: ['Qirmizi', 'Yasil', 'Mavi'] 

      Reng  Qiymet  Reng_kod
0  Qirmizi      10         0
1    Yasil      20         1
2     Mavi      15         2
3    Yasil      22         1
4  Qirmizi      11         0

NaN ile factori

## 2. SimpleImputer

### SimpleImputer nedir ve ne üçün istifade olunur?
Real datasetlerde çox vaxt bezi hüceyreler boş olur (NaN). Ekser ML modelleri NaN qebul etmir ve xeta verir. `SimpleImputer` (sklearn.impute) çatışmayan deyerleri sabit bir qayda ile doldurmaq üçün istifade olunur. Qayda `strategy` parametri ile seçilir. Obyekt `fit()` zamanı her sütun üçün doldurma deyerini (ortalama, median ve s.) öyrenir, `transform()` zamanı ise boşluqları bu deyerlerle doldurur.

### Strategiyalar
`mean` sütunun ortalamasını yazır, yalnız ededi sütunlar üçündür ve kenar deyerlere (outlier) hessasdır. `median` sütunun ortanca deyerini yazır, ededi sütunlar üçündür ve outlier-lere davamlıdır, ona göre maaş kimi çarpıq paylanmış deyerlerde daha yaxşıdır. `most_frequent` en çox tekrarlanan deyeri yazır, hem ededi, hem kateqorik sütunlarda işleyir. `constant` özümüzün verdiyimiz sabit deyeri yazır (`fill_value` parametri ile), meselen eded üçün 0, metn üçün "Namelum".


In [3]:
data = pd.DataFrame({
    "Yas":   [25, 30, np.nan, 40, 35, np.nan],
    "Maas":  [1000, 1500, 1200, np.nan, 5000, 1300],
    "Sheher": ["Baki", "Gence", np.nan, "Baki", "Baki", "Sumqayit"]
})
print("Boshluqlar:")
print(data.isna().sum(), "\n")

num_cols = ["Yas", "Maas"]

# 1) mean
imp_mean = SimpleImputer(strategy="mean")
print("mean:\n", pd.DataFrame(imp_mean.fit_transform(data[num_cols]), columns=num_cols), "\n")
print("Oyrenilen ortalamalar:", imp_mean.statistics_, "\n")

# 2) median
imp_med = SimpleImputer(strategy="median")
print("median:\n", pd.DataFrame(imp_med.fit_transform(data[num_cols]), columns=num_cols), "\n")

# 3) most_frequent (kateqorik sutun)
imp_freq = SimpleImputer(strategy="most_frequent")
data["Sheher_dolu"] = imp_freq.fit_transform(data[["Sheher"]]).ravel()
print("most_frequent:\n", data[["Sheher", "Sheher_dolu"]], "\n")

# 4) constant
imp_const = SimpleImputer(strategy="constant", fill_value=0)
print("constant (0):\n", pd.DataFrame(imp_const.fit_transform(data[num_cols]), columns=num_cols), "\n")

imp_const_txt = SimpleImputer(strategy="constant", fill_value="Namelum")
print("constant (metn):", imp_const_txt.fit_transform(data[["Sheher"]]).ravel())

# Alternativ: pandas ile
print("\npandas fillna:", data["Yas"].fillna(data["Yas"].mean()).tolist())

Boshluqlar:
Yas       2
Maas      1
Sheher    1
dtype: int64 

mean:
     Yas    Maas
0  25.0  1000.0
1  30.0  1500.0
2  32.5  1200.0
3  40.0  2000.0
4  35.0  5000.0
5  32.5  1300.0 

Oyrenilen ortalamalar: [  32.5 2000. ] 

median:
     Yas    Maas
0  25.0  1000.0
1  30.0  1500.0
2  32.5  1200.0
3  40.0  1300.0
4  35.0  5000.0
5  32.5  1300.0 

most_frequent:
      Sheher Sheher_dolu
0      Baki        Baki
1     Gence       Gence
2       NaN        Baki
3      Baki        Baki
4      Baki        Baki
5  Sumqayit    Sumqayit 

constant (0):
     Yas    Maas
0  25.0  1000.0
1  30.0  1500.0
2   0.0  1200.0
3  40.0     0.0
4  35.0  5000.0
5   0.0  1300.0 

constant (metn): ['Baki' 'Gence' 'Namelum' 'Baki' 'Baki' 'Sumqayit']

pandas fillna: [25.0, 30.0, 32.5, 40.0, 35.0, 32.5]


In [4]:
# Train/test ile duzgun istifade (data leakage olmadan)
train = pd.DataFrame({"x": [1.0, 2.0, np.nan, 4.0]})
test  = pd.DataFrame({"x": [np.nan, 10.0]})

imp = SimpleImputer(strategy="mean")
imp.fit(train)                      # yalniz train uzerinde oyren
print("Train ortalamasi:", imp.statistics_)
print("Test netice:", imp.transform(test).ravel())

Train ortalamasi: [2.33333333]
Test netice: [ 2.33333333 10.        ]


## 3. Regression Metrics

Regressiyada model davamlı (eded) deyer proqnozlaşdırır. Proqnozun heqiqi deyerden ne qeder uzaq olduğunu ölçmek üçün metrikalardan istifade edilir. Aşağıda $y_i$ heqiqi deyer, $\hat{y}_i$ proqnoz, $\bar{y}$ heqiqi deyerlerin ortalaması, $n$ ise nümune sayıdır.

### MAE - Mean Absolute Error
Proqnozlarla heqiqi deyerler arasındakı mütleq ferqlerin ortalamasını ölçür.

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

Netice hedef deyişenin öz vahidindedir (meselen manat). Meselen MAE = 5 demek, model orta hesabla 5 manat sehv edir. Kiçik olması yaxşıdır, ideal deyer 0-dır. Bütün xetalara eyni çeki verir, ona göre outlier-lere az hessasdır ve şerhi en sade metrikadır.

### MSE - Mean Squared Error
Xetaların kvadratlarının ortalamasını ölçür.

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2$$

Xetalar kvadrata yükseldildiyi üçün böyük sehvler çox cezalandırılır, ona göre outlier-lere çox hessasdır. Vahidi hedefin vahidinin kvadratıdır (manat²), bu sebebden birbaşa şerh etmek çetindir. Diferensiallanan olduğu üçün modelin öyrenmesi zamanı itki funksiyası kimi geniş işledilir.

### RMSE - Root Mean Squared Error
MSE-nin kvadrat kökü.

$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

MSE-nin üstünlüyünü (böyük xetalara hessaslıq) saxlayır, amma vahidi yene hedefle eynidir, ona göre MAE kimi şerh olunur. Adeten RMSE ≥ MAE olur. Ferq ne qeder böyükdürse, datada bir neçe böyük xeta var demekdir.

### R² - Determination Coefficient
Modelin hedef deyişenin deyişkenliyinin ne qeder hissesini izah etdiyini gösterir.

$$R^2 = 1-\frac{\sum (y_i-\hat{y}_i)^2}{\sum (y_i-\bar{y})^2} = 1-\frac{SS_{res}}{SS_{tot}}$$

Deyer 1-e yaxın olduqca model daha yaxşıdır: 1 mükemmel proqnoz, 0 modelin sadece ortalamanı proqnoz etmekden ferqlenmediyi, menfi deyer ise modelin ortalamadan da pis olduğu menasına gelir. Meselen R² = 0.85 demek, hedefdeki deyişkenliyin 85%-i model terefinden izah olunur. Diger üç metrikadan ferqi odur ki, o xeta deyil, nisbi keyfiyyet göstericisidir ve vahidsizdir, ona göre müxtelif datasetlerde modelleri müqayise etmek mümkündür.

### Metrikaların müqayisesi ve seçim
Outlier-ler vacib deyilse ve sade şerh lazımdırsa MAE, böyük xetaları xüsusi cezalandırmaq lazımdırsa RMSE/MSE, modelin ümumi izahetme gücünü göstermek üçün R² seçilir. Praktikada adeten bir neçesi birlikde bildirilir.

In [5]:
y_true = np.array([100, 150, 200, 250, 300])
y_pred = np.array([110, 140, 195, 270, 290])

# --- Elle hesablama ---
err = y_true - y_pred
mae  = np.mean(np.abs(err))
mse  = np.mean(err ** 2)
rmse = np.sqrt(mse)
ss_res = np.sum(err ** 2)
ss_tot = np.sum((y_true - y_true.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"Elle   -> MAE={mae:.3f}  MSE={mse:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")

# --- sklearn ile ---
print(f"sklearn-> MAE={mean_absolute_error(y_true, y_pred):.3f}  "
      f"MSE={mean_squared_error(y_true, y_pred):.3f}  "
      f"RMSE={np.sqrt(mean_squared_error(y_true, y_pred)):.3f}  "
      f"R2={r2_score(y_true, y_pred):.4f}")

Elle   -> MAE=11.000  MSE=145.000  RMSE=12.042  R2=0.9710
sklearn-> MAE=11.000  MSE=145.000  RMSE=12.042  R2=0.9710


In [6]:
# Outlier-in metrikalara tesiri: bir boyuk xeta elave edirik
y_true2 = np.array([100, 150, 200, 250, 300])
y_pred2 = np.array([110, 140, 195, 270, 500])   # sonuncu proqnoz cox sehvdir

def hesabat(ad, yt, yp):
    mse = mean_squared_error(yt, yp)
    print(f"{ad:12s} MAE={mean_absolute_error(yt, yp):7.2f}  MSE={mse:9.2f}  "
          f"RMSE={np.sqrt(mse):7.2f}  R2={r2_score(yt, yp):7.3f}")

hesabat("Normal", y_true, y_pred)
hesabat("Outlier-li", y_true2, y_pred2)
print("\nDiqqet: MSE/RMSE MAE-ye nisbeten daha cox artdi.")

# R2 menfi ola biler: ortalamadan da pis model
y_bad = np.array([300, 250, 200, 150, 100])
print("\nTers proqnozun R2-si:", round(r2_score(y_true, y_bad), 3))

Normal       MAE=  11.00  MSE=   145.00  RMSE=  12.04  R2=  0.971
Outlier-li   MAE=  49.00  MSE=  8125.00  RMSE=  90.14  R2= -0.625

Diqqet: MSE/RMSE MAE-ye nisbeten daha cox artdi.

Ters proqnozun R2-si: -3.0


## 4. Hamısının birlikde istifadesi (mini nümune)
Son olaraq bütün mövzuları bir pipeline-da birleşdiririk: çatışan deyerleri SimpleImputer ile doldururuq, kateqorik sütunu OneHotEncoder ile kodlayırıq, Linear Regression modelini öyredib metrikalarla qiymetlendiririk.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
n = 200
X = pd.DataFrame({
    "sahe":  rng.uniform(40, 150, n),
    "otaq":  rng.integers(1, 6, n).astype(float),
    "rayon": rng.choice(["Nesimi", "Yasamal", "Narimanov"], n),
})
y = 800 * X["sahe"] + 5000 * X["otaq"] + X["rayon"].map({"Nesimi": 20000, "Yasamal": 15000, "Narimanov": 8000}) \
    + rng.normal(0, 5000, n)

# Tesadufi boshluqlar yaradaq
X.loc[rng.choice(n, 15, replace=False), "sahe"] = np.nan
X.loc[rng.choice(n, 10, replace=False), "rayon"] = np.nan

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

num_pipe = SimpleImputer(strategy="median")
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])
pre = ColumnTransformer([
    ("num", num_pipe, ["sahe", "otaq"]),
    ("cat", cat_pipe, ["rayon"]),
])
model = Pipeline([("pre", pre), ("reg", LinearRegression())])

model.fit(X_tr, y_tr)
pred = model.predict(X_te)

mse = mean_squared_error(y_te, pred)
print(f"MAE  : {mean_absolute_error(y_te, pred):,.0f}")
print(f"MSE  : {mse:,.0f}")
print(f"RMSE : {np.sqrt(mse):,.0f}")
print(f"R2   : {r2_score(y_te, pred):.4f}")

MAE  : 6,292
MSE  : 92,342,814
RMSE : 9,610
R2   : 0.8747
